# Дообучение BERT на RuCoLA

**Задача** — бинарная классификация: приемлемо ли предложение русского языка с точки зрения грамматики
(RuCoLA — Russian Corpus of Linguistic Acceptability, аналог английской CoLA).

План работы:
1. Окружение и данные (`in_domain_train.csv` → train/val 80/20, `in_domain_dev.csv` → test)
2. Токенизация (`datasets` + токенизатор `sberbank-ai/ruBert-base`)
3. Обучение `sberbank-ai/ruBert-base`
4. Оценка качества на test + анализ ошибок
5. Обучение `cointegrated/rubert-tiny2` + сравнение моделей
6. Подбор гиперпараметров (learning rate, batch size, warmup ratio)

Оборудование: Apple M1 Max, устройство `mps` (Metal Performance Shaders).


In [1]:
import itertools
import json
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# на Mac с Apple Silicon доступен ускоритель mps
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | device: {DEVICE}")


/Users/rurik/repos/otus/bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 4.57.6 | torch 2.14.1 | device: mps


## 1. Окружение и данные

Установка: `pip install transformers datasets accelerate` (плюс `torch`, `scikit-learn`, `pandas`).

Датасет [RuCoLA](https://github.com/RussianNLP/RuCoLA):
- `in_domain_train.csv` — обучающая выборка, внутри делим её на **train (80%)** и **val (20%)** со стратификацией по классу;
- `in_domain_dev.csv` — выступает в роли **test**.

Оставляем только столбцы `sentence` и `acceptable` (целевая переменная: 1 — грамматически приемлемое предложение, 0 — ошибочное).


In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RUCOLA_URLS = {
    "in_domain_train.csv": "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_train.csv",
    "in_domain_dev.csv": "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_dev.csv",
}
for name, url in RUCOLA_URLS.items():  # скачиваем один раз
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(url, DATA_DIR / name)

full_train = pd.read_csv(DATA_DIR / "in_domain_train.csv")
test_df = pd.read_csv(DATA_DIR / "in_domain_dev.csv")

# оставляем только нужные столбцы: sentence и acceptable (целевая)
full_train = full_train[["sentence", "acceptable"]]
test_df = test_df[["sentence", "acceptable"]]

# train (80%) / val (20%): стратификация по целевой, чтобы сохранить баланс классов
train_df, val_df = train_test_split(
    full_train, test_size=0.2, stratify=full_train["acceptable"], random_state=SEED
)

print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: доля приемлемых предложений = {df['acceptable'].mean():.2%}")


train: 6295, val: 1574, test: 983
train: доля приемлемых предложений = 74.52%
val: доля приемлемых предложений = 74.52%
test: доля приемлемых предложений = 74.57%


## 2. Токенизация данных

Переводим выборки в формат `datasets.DatasetDict` и токенизируем токенизатором `sberbank-ai/ruBert-base`.
Целевую колонку переименовываем в `label` — именно её ждёт `AutoModelForSequenceClassification`.


In [3]:
BASE_MODEL = "sberbank-ai/ruBert-base"
TINY_MODEL = "cointegrated/rubert-tiny2"

# прикидываем длину предложений в токенах, чтобы выбрать max_length
tmp_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
lens = [len(ids) for ids in tmp_tok(list(full_train["sentence"]))["input_ids"]]
print("длина в токенах, перцентили 50/95/99/100:",
      np.percentile(lens, [50, 95, 99, 100]).round(1))


длина в токенах, перцентили 50/95/99/100: [12. 28. 36. 45.]


In [4]:
MAX_LEN = 64  # с запасом покрывает все предложения (99-й перцентиль << 64)


def build_datasets(model_name: str):
    """Токенизирует train/val/test токенизатором указенной модели."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "val": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }).rename_column("acceptable", "label")  # стандартное имя целевой колонки
    ds = ds.map(
        lambda batch: tokenizer(batch["sentence"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["sentence"],
    )
    return tokenizer, ds


base_tokenizer, base_ds = build_datasets(BASE_MODEL)
print(base_ds)
print(base_ds["train"][0])


Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:  95%|█████████▌| 6000/6295 [00:00<00:00, 31011.60 examples/s]

Map: 100%|██████████| 6295/6295 [00:00<00:00, 31582.37 examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map: 100%|██████████| 1574/1574 [00:00<00:00, 76649.65 examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

Map: 100%|██████████| 983/983 [00:00<00:00, 82251.10 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6295
    })
    val: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1574
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 983
    })
})
{'label': 1, 'input_ids': [101, 114, 5962, 121, 5141, 49850, 6652, 15520, 672, 9095, 126, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 3. Обучение `sberbank-ai/ruBert-base`

Гиперпараметры (в рамках, заданных заданием):
- оптимизатор **AdamW** (`adamw_torch` — оптимизатор `Trainer` по умолчанию);
- `batch_size = 16` (диапазон 8–32);
- `learning_rate = 2e-5` (диапазон 1e-5 — 5e-5);
- до 4 эпох, оценка на val после каждой эпохи;
- обучение продолжаем, пока val loss не выйдет на плато: `EarlyStoppingCallback` с patience = 2
  (останавливаемся после двух эпох подряд без улучшения val loss),
  лучшая по val loss модель восстанавливается в конце (`load_best_model_at_end`).

Для контроля времени каждая эпоха замеряется колбэком `RunTracker`; метрики дублируются в `logs/metrics.jsonl`.


In [5]:
LOG_FILE = Path("logs/metrics.jsonl")  # дублируем метрики сюда для мониторинга
LOG_FILE.parent.mkdir(exist_ok=True)


def compute_metrics(eval_pred):
    """Accuracy, F1 и MCC по logits."""
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "acc": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "mcc": matthews_corrcoef(labels, preds),
    }


class RunTracker(TrainerCallback):
    """Замеряет длительность каждой эпохи; метрики валидации добирает из on_log
    (в on_epoch_end они ещё не готовы) и пишет их в jsonl-лог.
    После trainer.train() сбор отключается (tracker.capture = False), чтобы финальный
    evaluate() лучшей модели не затирал метрики последней эпохи."""

    def __init__(self, run_name: str):
        self.run_name = run_name
        self._t0 = None
        self.capture = True
        self.history = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self._t0 = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        self.history.append({
            "run": self.run_name,
            "epoch": round(state.epoch, 2),
            "epoch_time_s": round(time.time() - self._t0, 1),
        })

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or not self.history or not self.capture:
            return
        rec = self.history[-1]
        if "eval_loss" in logs:  # метрики валидации закончившейся эпохи
            for key, value in logs.items():
                if isinstance(value, (int, float)):
                    rec[key.removeprefix("eval_")] = round(float(value), 4)
            with LOG_FILE.open("a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        elif "loss" in logs:  # train loss эпохи
            rec["train_loss"] = round(float(logs["loss"]), 4)


def train_model(model_name, train_ds, eval_ds, *, run_name, lr=2e-5, batch_size=16,
                epochs=4, warmup_ratio=0.1, weight_decay=0.01, patience=2):
    """Обучает модель классификации; возвращает (trainer, summary с таймингами и метриками)."""
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "unacceptable", 1: "acceptable"},
        label2id={"unacceptable": 0, "acceptable": 1},
    )

    args = TrainingArguments(
        output_dir=f"outputs/{run_name}",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        warmup_ratio=warmup_ratio,        # разогрев lr на первых шагах
        weight_decay=weight_decay,
        eval_strategy="epoch",            # оценка на val после каждой эпохи
        save_strategy="epoch",
        save_total_limit=1,
        save_only_model=True,             # чекпойнты без состояния оптимизатора — экономим диск
        load_best_model_at_end=True,      # в конце возвращаем лучшую по val loss модель
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
        disable_tqdm=True,
        dataloader_pin_memory=False,      # на mps пининг памяти не нужен
    )

    tracker = RunTracker(run_name)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),  # динамический паддинг по батчу
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience), tracker],
    )

    t_start = time.time()
    trainer.train()
    total_s = time.time() - t_start

    tracker.capture = False  # дальше идут финальные вызовы — историю эпох не трогаем
    val_metrics = trainer.evaluate()  # метрики лучшей (восстановленной) модели на val
    best = min(tracker.history, key=lambda r: r.get("loss", float("inf")))

    summary = {
        "run": run_name,
        "model": model_name,
        "lr": lr,
        "batch_size": batch_size,
        "warmup_ratio": warmup_ratio,
        "params_m": round(model.num_parameters() / 1e6, 1),
        "epochs_trained": len(tracker.history),
        "best_epoch": best["epoch"],
        "epoch_times_s": [r["epoch_time_s"] for r in tracker.history],
        "mean_epoch_time_s": round(float(np.mean([r["epoch_time_s"] for r in tracker.history])), 1),
        "total_train_time_s": round(total_s, 1),
        "val": {k: round(float(v), 4) for k, v in val_metrics.items() if isinstance(v, (int, float))},
        "history": tracker.history,
    }
    with LOG_FILE.open("a") as f:
        f.write(json.dumps({"run": run_name, "event": "summary",
                            **{k: v for k, v in summary.items() if k != "history"}},
                           ensure_ascii=False) + "\n")
    return trainer, summary


In [6]:
trainer_base, summary_base = train_model(
    BASE_MODEL, base_ds["train"], base_ds["val"], run_name="rubert-base"
)

display(pd.DataFrame(summary_base["history"]))
print(json.dumps({k: v for k, v in summary_base.items() if k != "history"},
                 ensure_ascii=False, indent=2))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sberbank-ai/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.5478, 'grad_norm': 11.129724502563477, 'learning_rate': 1.668547249647391e-05, 'epoch': 1.0}


{'eval_loss': 0.48700493574142456, 'eval_acc': 0.7833545108005082, 'eval_f1': 0.8695984703632887, 'eval_mcc': 0.3280929138124646, 'eval_runtime': 2.414, 'eval_samples_per_second': 652.036, 'eval_steps_per_second': 10.356, 'epoch': 1.0}


{'loss': 0.3617, 'grad_norm': 3.7671968936920166, 'learning_rate': 1.1128349788434416e-05, 'epoch': 2.0}


{'eval_loss': 0.5044313073158264, 'eval_acc': 0.7922490470139771, 'eval_f1': 0.8694610778443114, 'eval_mcc': 0.38945871525805265, 'eval_runtime': 2.4573, 'eval_samples_per_second': 640.531, 'eval_steps_per_second': 10.174, 'epoch': 2.0}


...

{'eval_loss': 0.7188529968261719, 'eval_acc': 0.7998729351969505, 'eval_f1': 0.8720032507110931, 'eval_mcc': 0.4278072389294639, 'eval_runtime': 2.4427, 'eval_samples_per_second': 644.375, 'eval_steps_per_second': 10.235, 'epoch': 3.0}


{'train_runtime': 736.9648, 'train_samples_per_second': 34.167, 'train_steps_per_second': 2.139, 'train_loss': 0.3670836705241712, 'epoch': 3.0}


{'eval_loss': 0.48700493574142456, 'eval_acc': 0.7833545108005082, 'eval_f1': 0.8695984703632887, 'eval_mcc': 0.3280929138124646, 'eval_runtime': 2.5286, 'eval_samples_per_second': 622.473, 'eval_steps_per_second': 9.887, 'epoch': 3.0}


,run,epoch,epoch_time_s,train_loss,loss,acc,f1,mcc,runtime,samples_per_second,steps_per_second
0,rubert-base,1.0,241.2,0.5478,0.4870,0.7834,0.8696,0.3281,2.4140,652.036,10.356
1,rubert-base,2.0,242.0,0.3617,0.5044,0.7922,0.8695,0.3895,2.4573,640.531,10.174
2,rubert-base,3.0,243.1,0.1918,0.7189,0.7999,0.8720,0.4278,2.4427,644.375,10.235


{
  "run": "rubert-base",
  "model": "sberbank-ai/ruBert-base",
  "lr": 2e-05,
  "batch_size": 16,
  "warmup_ratio": 0.1,
  "params_m": 178.3,
  "epochs_trained": 3,
  "best_epoch": 1.0,
  "epoch_times_s": [
    241.2,
    242.0,
    243.1
  ],
  "mean_epoch_time_s": 242.1,
  "total_train_time_s": 737.1,
  "val": {
    "eval_loss": 0.487,
    "eval_acc": 0.7834,
    "eval_f1": 0.8696,
    "eval_mcc": 0.3281,
    "eval_runtime": 2.5286,
    "eval_samples_per_second": 622.473,
    "eval_steps_per_second": 9.887,
    "epoch": 3.0
  }
}


### Анализ обучения ruBert-base

Динамика по эпохам (val):

| эпоха | train loss | val loss | val acc | val F1 | val MCC | время, с |
|---|---|---|---|---|---|---|
| 1 | 0.548 | **0.487** | 0.783 | 0.870 | 0.328 | 241 |
| 2 | 0.362 | 0.504 | 0.792 | 0.869 | 0.389 | 242 |
| 3 | 0.192 | 0.719 | 0.800 | 0.872 | 0.428 | 243 |

**Время обучения.** Одна эпоха ≈ **242 с (~4 мин)**, стабильна от эпохи к эпохе. Всего обучение
заняло **737 с ≈ 12.3 мин** (6 295 примеров, batch 16 → 394 шага/эпоху, fp32 на mps).

**Эпох до плато.** Val loss минимален уже после **1-й эпохи** (0.487), дальше только растёт
(0.504 → 0.719) — классическая картина быстрого переобучения BERT на небольшой выборке.
Ранняя остановка (patience = 2) сработала после **3-й эпохи**; лучшая по val loss модель
(эпоха 1) восстановлена через `load_best_model_at_end`.

**Наблюдение:** val MCC при этом продолжал расти (0.328 → 0.428). Loss и метрика качества
расходятся: с переобучением модель становится «самоуверенной» (loss штрафует уверенные ошибки
сильнее), тогда как количество верных ответов ещё растёт. Чекпойнт мы выбираем по val loss —
в согласии с заданием (критерий останова — плато val loss).

## 4. Оценка качества на test

Тест — `in_domain_dev.csv`. Считаем все три метрики (accuracy, F1, MCC), чтобы обосновать выбор главной.


In [7]:
def predict(trainer, ds):
    """Метрики, предсказанные классы и вероятности класса acceptable на датасете."""
    out = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy()
    preds = out.predictions.argmax(-1)
    metrics = compute_metrics((out.predictions, out.label_ids))
    return metrics, preds, probs


base_test_metrics, base_preds, base_probs = predict(trainer_base, base_ds["test"])
print("ruBert-base на test (in_domain_dev):",
      {k: round(v, 4) for k, v in base_test_metrics.items()})


ruBert-base на test (in_domain_dev): {'acc': 0.7965, 'f1': 0.8775, 'mcc': 0.3814}


### Какая метрика показательнее

Классы в RuCoLA **несбалансированы**: приемлемых предложений ≈ 74.5% во всех выборках.
Поэтому:

- **Accuracy** обманчива: константный классификатор «всё приемлемо» уже даёт **acc = 74.6%**,
  а наша модель — 79.7%: разница кажется скромной, хотя модель реально решает задачу.
- **F1** (по классу «приемлемо») тоже завышена дисбалансом: у константного классификатора
  F1 = **0.854**, у ruBert-base — 0.877. Кроме того, F1 зависит от порога 0.5 и игнорирует
  ошибки по отрицательному классу.
- **MCC** (корреляция Мэтьюса) использует все четыре ячейки матрицы ошибок, симметричен
  к классам и почти не реагирует на дисбаланс: у константного классификатора MCC = **0**,
  у нашей модели — 0.38. Именно MCC — официальная метрика бенчмарков CoLA/RuCoLA.

**Вывод: для этой задачи показательнее всего MCC**; accuracy и F1 приведены для полноты.
Test-результат ruBert-base: **acc 0.797, F1 0.877, MCC 0.381**.

In [8]:
def show_errors(df, preds, probs, n=10):
    """n случайных ошибок модели: предложение, истинный класс, предсказание, уверенность."""
    res = df.assign(pred=preds, p_acceptable=probs.round(3))
    errors = res[res["acceptable"] != res["pred"]]
    print(f"ошибок: {len(errors)} из {len(res)} ({len(errors) / len(res):.1%})")
    return errors.sample(n, random_state=SEED).reset_index(drop=True)


show_errors(test_df, base_preds, base_probs)


ошибок: 200 из 983 (20.3%)


,sentence,acceptable,pred,p_acceptable
0,"И пошлите кого-нибудь к Фу Чану, чтоб захватил...",0,1,0.851
1,Девушек на сцене не пело.,0,1,0.807
2,"Иван войдя, он поздоровался.",0,1,0.835
3,В результате этого он получил оспу.,0,1,0.819
4,"Едва Олеся, закончив все приготовления на кухн...",0,1,0.601
5,Она медленно становится бледной.,0,1,0.731
6,"Я смотрю на детей, бросающих друг другу мяч.",1,0,0.409
7,Через несколько дней после ссоры Дубровский по...,0,1,0.845
8,В седьмой день седьмого лунного месяца по улиц...,0,1,0.556
9,Доклад написан уже час.,0,1,0.886


### Анализ ошибок ruBert-base

Модель ошиблась на **200 из 983** предложений test (20.3%). В показанной выборке из 10 ошибок
**9 — ложные «приемлемо»** (пропуск ошибки), 1 — ложное «неприемлемо». Это ожидаемый сдвиг:
класс «приемлемо» в 3 раза крупнее, и порог 0.5 смещает модель к нему.

Возможные причины:

1. **Тонкие лингвистические явления вместо явных ошибок.** Предложения синтаксически гладкие,
   а нарушение — в семантике/аспектуальности или редкой конструкции: *«Она медленно становится
   бледной»* (норма требует видового глагола *бледнеет*), *«Доклад написан уже час»* (вид/
   время), *«Девушек на сцене не пело»* (спорное согласование), *«Иван войдя, он
   поздоровался»* (деепричастный оборот с дублированным подлежащим).
2. **Шум и спорность разметки.** Часть примеров тяжелы даже для носителей: *«В результате
   этого он получил оспу»* помечена как неприемлемая, хотя выглядит нормально; единственную
   ошибку «в другую сторону» — *«Я смотрю на детей, бросающих друг другу мяч»* — модель
   забраковала, и тоже не зря: сочетание *бросающих друг другу* сомнительно.
3. **Дисбаланс классов** → модель недообучена именно на классе ошибок (MCC < 0.4).
4. **Уверенность ошибок умеренная** (p ≈ 0.56–0.89): многие ошибки лежат у порога —
   подбор порога или взвешивание классов могли бы сдвинуть баланс полнота/точность.

## 5. «Маленький BERT»: `cointegrated/rubert-tiny2`

Меняем модель на `cointegrated/rubert-tiny2` (в ~6 раз меньше ruBert-base) и **обязательно заново токенизируем** данные —
у tiny2 собственный словарь. Обучаем по той же процедуре, что и в п. 3.


In [9]:
# у tiny2 свой токенизатор/словарь — ретокенизируем данные
tiny_tokenizer, tiny_ds = build_datasets(TINY_MODEL)
print(f"размер словаря: ruBert-base = {len(base_tokenizer)}, rubert-tiny2 = {len(tiny_tokenizer)}")


Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map: 100%|██████████| 6295/6295 [00:00<00:00, 95855.27 examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map: 100%|██████████| 1574/1574 [00:00<00:00, 86194.83 examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

Map: 100%|██████████| 983/983 [00:00<00:00, 85005.07 examples/s]

размер словаря: ruBert-base = 120138, rubert-tiny2 = 83828


In [10]:
trainer_tiny, summary_tiny = train_model(
    TINY_MODEL, tiny_ds["train"], tiny_ds["val"], run_name="rubert-tiny2"
)

display(pd.DataFrame(summary_tiny["history"]))
print(json.dumps({k: v for k, v in summary_tiny.items() if k != "history"},
                 ensure_ascii=False, indent=2))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.5776, 'grad_norm': 1.7524940967559814, 'learning_rate': 1.668547249647391e-05, 'epoch': 1.0}


{'eval_loss': 0.5439013242721558, 'eval_acc': 0.7452350698856417, 'eval_f1': 0.854022570076447, 'eval_mcc': 0.0, 'eval_runtime': 0.4842, 'eval_samples_per_second': 3250.729, 'eval_steps_per_second': 51.632, 'epoch': 1.0}


{'loss': 0.5289, 'grad_norm': 1.5261826515197754, 'learning_rate': 1.1128349788434416e-05, 'epoch': 2.0}


{'eval_loss': 0.5333604216575623, 'eval_acc': 0.7541296060991105, 'eval_f1': 0.8552188552188552, 'eval_mcc': 0.17322678800170924, 'eval_runtime': 0.3283, 'eval_samples_per_second': 4794.026, 'eval_steps_per_second': 76.144, 'epoch': 2.0}


...

{'loss': 0.492, 'grad_norm': 3.194293975830078, 'learning_rate': 1.4104372355430185e-08, 'epoch': 4.0}


{'eval_loss': 0.5333797931671143, 'eval_acc': 0.7592121982210928, 'eval_f1': 0.8555089592070149, 'eval_mcc': 0.2241222603640231, 'eval_runtime': 0.3395, 'eval_samples_per_second': 4635.768, 'eval_steps_per_second': 73.63, 'epoch': 4.0}
{'train_runtime': 176.3124, 'train_samples_per_second': 142.815, 'train_steps_per_second': 8.939, 'train_loss': 0.5268184932960471, 'epoch': 4.0}


{'eval_loss': 0.5333604216575623, 'eval_acc': 0.7541296060991105, 'eval_f1': 0.8552188552188552, 'eval_mcc': 0.17322678800170924, 'eval_runtime': 0.3673, 'eval_samples_per_second': 4285.628, 'eval_steps_per_second': 68.069, 'epoch': 4.0}


,run,epoch,epoch_time_s,train_loss,loss,acc,f1,mcc,runtime,samples_per_second,steps_per_second
0,rubert-tiny2,1.0,44.6,0.5776,0.5439,0.7452,0.8540,0.0000,0.4842,3250.729,51.632
1,rubert-tiny2,2.0,43.6,0.5289,0.5334,0.7541,0.8552,0.1732,0.3283,4794.026,76.144
2,rubert-tiny2,3.0,44.1,0.5088,0.5342,0.7484,0.8443,0.2274,0.3476,4528.414,71.925
3,rubert-tiny2,4.0,41.8,0.4920,0.5334,0.7592,0.8555,0.2241,0.3395,4635.768,73.630


{
  "run": "rubert-tiny2",
  "model": "cointegrated/rubert-tiny2",
  "lr": 2e-05,
  "batch_size": 16,
  "warmup_ratio": 0.1,
  "params_m": 29.2,
  "epochs_trained": 4,
  "best_epoch": 2.0,
  "epoch_times_s": [
    44.6,
    43.6,
    44.1,
    41.8
  ],
  "mean_epoch_time_s": 43.5,
  "total_train_time_s": 176.6,
  "val": {
    "eval_loss": 0.5334,
    "eval_acc": 0.7541,
    "eval_f1": 0.8552,
    "eval_mcc": 0.1732,
    "eval_runtime": 0.3673,
    "eval_samples_per_second": 4285.628,
    "eval_steps_per_second": 68.069,
    "epoch": 4.0
  }
}


### Анализ обучения rubert-tiny2

Динамика по эпохам (val): loss 0.544 → 0.533 → 0.534 → 0.533 — **плато примерно со 2-й эпохи**,
но две эпохи подряд без улучшения в рамках лимита (4 эпохи) не наступило, поэтому были
обучены все 4.

| | ruBert-base | rubert-tiny2 | соотношение |
|---|---|---|---|
| параметры | 178.3 млн | 29.2 млн | 6.1× |
| словарь | 120 138 | 83 828 | 1.4× |
| секунд на эпоху | 242.1 | 43.5 | **5.6× быстрее** |
| эпох обучено | 3 | 4 | |
| полное время обучения | 737 с (12.3 мин) | 177 с (2.9 мин) | 4.2× быстрее |

**Скорость.** Эпоха tiny2 занимает **~43.5 с против ~242 с** у ruBert-base — в 5.6 раза
быстрее (при том что батчи и данные те же, выигрыш чисто от размера модели). До плато tiny2
шла дольше в эпохах (4 против 3), но в стенном времени обучение быстрее в 4.2 раза.

**Качество заметно ниже:** на 1-й эпохе tiny2 вообще выродилась в константный классификатор
(val MCC = 0), к концу val MCC 0.22–0.23 против 0.43 у base. Маленькой модели не хватает
ёмкости, и «включается» она медленнее.

## 6. Сравнение моделей

Оцениваем `rubert-tiny2` на test по процедуре из п. 4 и сравниваем с `ruBert-base`:
качество против скорость и размер.


In [11]:
tiny_test_metrics, tiny_preds, tiny_probs = predict(trainer_tiny, tiny_ds["test"])
print("rubert-tiny2 на test (in_domain_dev):",
      {k: round(v, 4) for k, v in tiny_test_metrics.items()})


def comparison_row(name, summary, test_metrics):
    return {
        "модель": name,
        "параметры, млн": summary["params_m"],
        "эпох (до плато)": summary["epochs_trained"],
        "сек / эпоху": summary["mean_epoch_time_s"],
        "полное время обучения, сек": summary["total_train_time_s"],
        "val loss": summary["val"]["eval_loss"],
        "val f1": summary["val"]["eval_f1"],
        "test acc": round(test_metrics["acc"], 4),
        "test f1": round(test_metrics["f1"], 4),
        "test mcc": round(test_metrics["mcc"], 4),
    }


comparison = pd.DataFrame([
    comparison_row("ruBert-base", summary_base, base_test_metrics),
    comparison_row("rubert-tiny2", summary_tiny, tiny_test_metrics),
])
comparison


rubert-tiny2 на test (in_domain_dev): {'acc': 0.7752, 'f1': 0.8671, 'mcc': 0.2847}


,модель,"параметры, млн",эпох (до плато),сек / эпоху,"полное время обучения, сек",val loss,val f1,test acc,test f1,test mcc
0,ruBert-base,178.3,3,242.1,737.1,0.4870,0.8696,0.7965,0.8775,0.3814
1,rubert-tiny2,29.2,4,43.5,176.6,0.5334,0.8552,0.7752,0.8671,0.2847


### Сравнение моделей

| модель | параметры | сек/эпоху | время обучения | test acc | test F1 | **test MCC** |
|---|---|---|---|---|---|---|
| ruBert-base | 178.3 млн | 242 | 12.3 мин | 0.797 | 0.877 | **0.381** |
| rubert-tiny2 | 29.2 млн | 43.5 | 2.9 мин | 0.775 | 0.867 | **0.285** |

- **ruBert-base** даёт +0.10 MCC на test — при том, что весь «разбег» метрики от константного
  классификатора (MCC = 0) до нашего результата — это существенный отрыв. Задача RuCoLA
  требует улавливать тонкие грамматические явления, и ёмкость энкодера здесь напрямую
  конвертируется в качество.
- **rubert-tiny2** — в 5.6 раза быстрее на эпоху, в 6 раз меньше (легче инференс, меньше
  память), но качество падает заметно; кроме того, маленькая модель капризнее к выбору lr
  (см. п. 7) и дольше «раскачивается».

**Заключение.** Для данной задачи (оффлайн-классификация, качество важнее latency) лучше
**ruBert-base**. rubert-tiny2 уместен, когда критичны скорость инференса/размер модели
или нужно много дешёвых экспериментов (что мы и делали в п. 7).

## 7. Подбор гиперпараметров

Исследуем влияние гиперпараметров на быстрой `rubert-tiny2` (полный grid на ruBert-base занял бы часы):

1. **Grid**: `learning_rate` ∈ {1e-5, 2e-5, 5e-5} × `batch_size` ∈ {8, 16, 32} — 9 запусков;
2. **Warmup**: при лучшей конфигурации грида сравниваем `warmup_ratio` ∈ {0.0, 0.1, 0.3}.

Каждый запуск — до 4 эпох с ранней остановкой по val loss (как в п. 3), метрика сравнения — val F1
(е же требует таблица в задании). Прогресс дублируется в `logs/grid_progress.csv`.


In [12]:
grid_rows = []
for lr, bs in itertools.product([1e-5, 2e-5, 5e-5], [8, 16, 32]):
    _, s = train_model(
        TINY_MODEL, tiny_ds["train"], tiny_ds["val"],
        run_name=f"tiny-lr{lr:g}-bs{bs}", lr=lr, batch_size=bs,
    )
    grid_rows.append({
        "lr": lr,
        "batch_size": bs,
        "epochs": s["epochs_trained"],
        "val_f1": round(s["val"]["eval_f1"], 4),
        "val_mcc": round(s["val"]["eval_mcc"], 4),
        "val_loss": round(s["val"]["eval_loss"], 4),
        "сек/эпоха": s["mean_epoch_time_s"],
        "notes": f"лучшая эпоха {s['best_epoch']}",
    })
    pd.DataFrame(grid_rows).to_csv("logs/grid_progress.csv", index=False)  # прогресс на случай перезапуска

grid_df = pd.DataFrame(grid_rows)
grid_df


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.5809, 'grad_norm': 1.6672860383987427, 'learning_rate': 8.337451464878221e-06, 'epoch': 1.0}


{'eval_loss': 0.5510157346725464, 'eval_acc': 0.7452350698856417, 'eval_f1': 0.854022570076447, 'eval_mcc': 0.0, 'eval_runtime': 0.4602, 'eval_samples_per_second': 3420.179, 'eval_steps_per_second': 54.323, 'epoch': 1.0}


{'loss': 0.5368, 'grad_norm': 1.6534149646759033, 'learning_rate': 5.559477585598306e-06, 'epoch': 2.0}


{'eval_loss': 0.538068413734436, 'eval_acc': 0.7509529860228716, 'eval_f1': 0.8542750929368029, 'eval_mcc': 0.14421739862158262, 'eval_runtime': 0.3324, 'eval_samples_per_second': 4735.815, 'eval_steps_per_second': 75.219, 'epoch': 2.0}


...

{'eval_loss': 0.5453898310661316, 'eval_acc': 0.7395171537484116, 'eval_f1': 0.835076427996782, 'eval_mcc': 0.23328899646344153, 'eval_runtime': 0.3422, 'eval_samples_per_second': 4599.796, 'eval_steps_per_second': 73.059, 'epoch': 3.0}


{'loss': 0.4313, 'grad_norm': 2.831012487411499, 'learning_rate': 7.052186177715092e-08, 'epoch': 4.0}


{'eval_loss': 0.5455566048622131, 'eval_acc': 0.7579415501905972, 'eval_f1': 0.85134607881389, 'eval_mcc': 0.25015089535713503, 'eval_runtime': 0.3419, 'eval_samples_per_second': 4603.645, 'eval_steps_per_second': 73.12, 'epoch': 4.0}
{'train_runtime': 95.2143, 'train_samples_per_second': 264.456, 'train_steps_per_second': 8.276, 'train_loss': 0.4966741455387948, 'epoch': 4.0}


{'eval_loss': 0.5268675088882446, 'eval_acc': 0.7655654383735705, 'eval_f1': 0.8588910133843213, 'eval_mcc': 0.2544481327264984, 'eval_runtime': 0.3747, 'eval_samples_per_second': 4200.236, 'eval_steps_per_second': 66.713, 'epoch': 4.0}


,lr,batch_size,epochs,val_f1,val_mcc,val_loss,сек/эпоха,notes
0,0.00001,8,4,0.8503,0.1850,0.5349,86.3,лучшая эпоха 3.0
1,0.00001,16,4,0.8539,0.1568,0.5369,43.1,лучшая эпоха 4.0
2,0.00001,32,4,0.8552,0.0821,0.5401,22.8,лучшая эпоха 4.0
3,0.00002,8,4,0.8548,0.1990,0.5307,85.9,лучшая эпоха 2.0
4,0.00002,16,4,0.8552,0.1732,0.5334,43.9,лучшая эпоха 2.0
5,0.00002,32,4,0.8545,0.2012,0.5329,23.0,лучшая эпоха 4.0
6,0.00005,8,4,0.8619,0.2812,0.5258,86.0,лучшая эпоха 2.0
7,0.00005,16,4,0.8604,0.2490,0.5250,44.1,лучшая эпоха 2.0
8,0.00005,32,4,0.8589,0.2544,0.5269,23.2,лучшая эпоха 2.0


In [13]:
best_row = grid_df.sort_values("val_f1", ascending=False).iloc[0]
print("лучшая конфигурация грида:", dict(best_row[["lr", "batch_size"]]))

warmup_rows = []
for wr in [0.0, 0.1, 0.3]:
    _, s = train_model(
        TINY_MODEL, tiny_ds["train"], tiny_ds["val"],
        run_name=f"tiny-warmup-{wr}", lr=float(best_row["lr"]),
        batch_size=int(best_row["batch_size"]), warmup_ratio=wr,
    )
    warmup_rows.append({
        "warmup_ratio": wr,
        "lr": best_row["lr"],
        "batch_size": best_row["batch_size"],
        "epochs": s["epochs_trained"],
        "val_f1": round(s["val"]["eval_f1"], 4),
        "val_mcc": round(s["val"]["eval_mcc"], 4),
        "notes": f"лучшая эпоха {s['best_epoch']}",
    })

pd.DataFrame(warmup_rows)


лучшая конфигурация грида: {'lr': np.float64(5e-05), 'batch_size': np.int64(8)}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.5448, 'grad_norm': 2.674051284790039, 'learning_rate': 3.7515883100381195e-05, 'epoch': 1.0}


{'eval_loss': 0.5369631052017212, 'eval_acc': 0.7560355781448539, 'eval_f1': 0.8547655068078669, 'eval_mcc': 0.19904482480954216, 'eval_runtime': 0.7262, 'eval_samples_per_second': 2167.539, 'eval_steps_per_second': 34.427, 'epoch': 1.0}


{'loss': 0.4876, 'grad_norm': 3.032654047012329, 'learning_rate': 2.5015883100381193e-05, 'epoch': 2.0}


...

{'eval_loss': 0.656578779220581, 'eval_acc': 0.7121982210927573, 'eval_f1': 0.8095838587641866, 'eval_mcc': 0.22132755395856887, 'eval_runtime': 0.3632, 'eval_samples_per_second': 4333.738, 'eval_steps_per_second': 68.833, 'epoch': 3.0}


{'loss': 0.3465, 'grad_norm': 2.9239563941955566, 'learning_rate': 2.2696323195642307e-08, 'epoch': 4.0}


{'eval_loss': 0.6973563432693481, 'eval_acc': 0.738881829733164, 'eval_f1': 0.8356657337065174, 'eval_mcc': 0.2221572852553714, 'eval_runtime': 0.4068, 'eval_samples_per_second': 3869.175, 'eval_steps_per_second': 61.454, 'epoch': 4.0}
{'train_runtime': 342.0369, 'train_samples_per_second': 73.618, 'train_steps_per_second': 9.204, 'train_loss': 0.4689278220888014, 'epoch': 4.0}


{'eval_loss': 0.5240026116371155, 'eval_acc': 0.7687420584498094, 'eval_f1': 0.8625377643504532, 'eval_mcc': 0.2569171909461599, 'eval_runtime': 0.3707, 'eval_samples_per_second': 4246.134, 'eval_steps_per_second': 67.442, 'epoch': 4.0}


,warmup_ratio,lr,batch_size,epochs,val_f1,val_mcc,notes
0,0.0,0.00005,8,4,0.8605,0.2858,лучшая эпоха 2.0
1,0.1,0.00005,8,4,0.8619,0.2812,лучшая эпоха 2.0
2,0.3,0.00005,8,4,0.8625,0.2569,лучшая эпоха 2.0


### Выводы по гиперпараметрам

**Learning rate — самый влиятельный параметр.** Для маленькой rubert-tiny2 оптимальным оказался
верх диапазона задания — **5e-5**: при нём val F1 = 0.859–0.862 при любом batch size против
0.850–0.855 при 1e-5. Маленькой модели нужен больший шаг обучения; при 1e-5 она за 4 эпохи
почти не отрывается от константного классификатора (val MCC 0.08–0.19).

**Batch size влияет слабо и неоднозначно** (различия ~0.003 F1 — в пределах шума): при 5e-5
лучший результат у bs 8 (0.8619), но bs 32 почти так же хорош при в 3.7 раза меньшем времени
эпохи (23 с против 86 с). Практический вывод: bs удобнее выбирать по скорости/памяти, а не
по качеству.

**Warmup практически не влияет** на итог (val F1 0.8605 / 0.8619 / 0.8625 при warmup 0.0 /
0.1 / 0.3) — при коротком обучении (4 эпохи) и малой модели эффект размывается. Отметим
воспроизводимость: запуск с warmup 0.1 совпал до последнего знака с grid-запуском той же
конфигурации.

**Итог:** важность параметров по убыванию — **lr ≫ batch size > warmup**. При этом у всех
конфигураций лучшая эпоха — 2-я (при 5e-5): высокая lr ускоряет и переобучение, поэтому
ранняя остановка обязательна.

## Итоговые выводы

1. **Данные.** RuCoLA in-domain: 6 295 train / 1 574 val / 983 test; классы несбалансированы
   (74.5% приемлемых) — это определило выбор метрики.
2. **ruBert-base** (AdamW, bs 16, lr 2e-5): val loss выходит на минимум уже после 1-й эпохи,
   ранняя остановка — после 3-й; эпоха ≈ 242 с, полное обучение ≈ 12 мин на Apple M1 Max (mps).
   Test: **MCC 0.381**, acc 0.797, F1 0.877.
3. **Метрика.** MCC — самая показательная: accuracy/F1 завышены дисбалансом (у константного
   классификатора acc 74.6% и F1 0.854 при MCC = 0).
4. **Ошибки** (20.3%) — в основном пропуски тонких грамматических/аспектуальных нарушений
   и спорных случаев разметки; модель смещена к частому классу.
5. **rubert-tiny2**: та же процедура, 4 эпохи, ~43.5 с/эпоху (в 5.6 раза быстрее), test
   MCC 0.285. Компромисс «ёмкость ↔ скорость» разрешается в пользу ruBert-base, если
   качество важнее latency.
6. **Гиперпараметры** (на tiny2): lr ≫ batch size > warmup; лучший lr 5e-5 (верх границы
   задания), batch size удобно выбирать из соображений скорости, warmup на коротком обучении
   почти не влияет.

Главная демонстрация задания: готовый предобученный BERT дообучается до сильного бейзлайна
на новом языке/задаче за **~12 минут на ноутбуке**, а «мини»-версия — за 3 минуты с потерей
~0.1 MCC.